# Exercise 06 — SOLUTION: Data Poisoning

## Background

See student notebook.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

transform = transforms.ToTensor()
train_data = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.net(x)

def train_model(model, dataset, epochs=3, lr=1e-3):
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn= nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    model.eval()

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=512)
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

# Train and evaluate clean baseline
baseline = SmallCNN()
train_model(baseline, train_data)
print(f"Clean baseline accuracy: {evaluate(baseline, test_data):.2%}")


## Backdoor Poisoning — Solution

In [ ]:
TRIGGER_SIZE  = 3
TRIGGER_VALUE = 1.0
TARGET_CLASS  = 0
POISON_RATE   = 0.10

def add_trigger(image):
    img = image.clone()
    img[0, :TRIGGER_SIZE, :TRIGGER_SIZE] = TRIGGER_VALUE
    return img

class PoisonedDataset(Dataset):
    def __init__(self, clean_dataset):
        rng = np.random.RandomState(42)
        self.samples = []
        for img, label in clean_dataset:
            if rng.random() < POISON_RATE:
                # SOLUTION
                self.samples.append((add_trigger(img), TARGET_CLASS))
            else:
                self.samples.append((img, label))
    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

poisoned_train = PoisonedDataset(train_data)
backdoor_model = SmallCNN()
train_model(backdoor_model, poisoned_train)

print(f"Backdoored model — clean accuracy: {evaluate(backdoor_model, test_data):.2%}")

triggered_test = [(add_trigger(img), label) for img, label in test_data if label != TARGET_CLASS]
triggered_ds   = torch.utils.data.TensorDataset(
    torch.stack([x for x, _ in triggered_test]),
    torch.tensor([y for _, y in triggered_test])
)
loader = DataLoader(triggered_ds, batch_size=512)
hits = total = 0
with torch.no_grad():
    for x, _ in loader:
        hits  += (backdoor_model(x).argmax(1) == TARGET_CLASS).sum().item()
        total += len(x)
print(f"Backdoored model — attack success rate (triggered→class {TARGET_CLASS}): {hits/total:.2%}")


## Label Flipping (pre-written)

In [ ]:
# Label flipping: vary poison rate and measure accuracy degradation
flip_rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
accs = []
for rate in flip_rates:
    flipped_data = []
    rng = np.random.RandomState(42)
    for img, label in train_data:
        if rng.random() < rate:
            new_label = rng.randint(0, 10)
            flipped_data.append((img, new_label))
        else:
            flipped_data.append((img, label))
    flipped_ds = torch.utils.data.TensorDataset(
        torch.stack([x for x,_ in flipped_data]),
        torch.tensor([y for _,y in flipped_data])
    )
    m = SmallCNN(); train_model(m, flipped_ds)
    accs.append(evaluate(m, test_data))
    print(f"  Flip rate {rate:.0%}: accuracy = {accs[-1]:.2%}")

plt.figure(figsize=(6,4))
plt.plot([r*100 for r in flip_rates], [a*100 for a in accs], 'o-')
plt.xlabel('Poison rate (%)'); plt.ylabel('Test accuracy (%)')
plt.title('Label Flipping: Accuracy vs Poison Rate')
plt.grid(True); plt.tight_layout(); plt.show()

## Visualization

In [ ]:
# Visualize: clean vs triggered images
fig, axes = plt.subplots(2, 5, figsize=(12,5))
for i in range(5):
    img, label = test_data[i]
    axes[0,i].imshow(img.squeeze(), cmap='gray')
    axes[0,i].set_title(f'Clean\nlabel={label}', fontsize=9); axes[0,i].axis('off')
    trig = add_trigger(img)
    axes[1,i].imshow(trig.squeeze(), cmap='gray')
    with torch.no_grad():
        pred = backdoor_model(trig.unsqueeze(0)).argmax(1).item()
    axes[1,i].set_title(f'Triggered\npred={pred}', fontsize=9); axes[1,i].axis('off')
plt.suptitle('Backdoor Attack: Clean vs Triggered (3×3 white square)', fontsize=12)
plt.tight_layout(); plt.show()
